In [ ]:
# Configuração para Google Colab (instalação automática de dependências extras)
import sys
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Instalando pacotes adicionais no Google Colab...")
    # ultralytics (YOLO) e easyocr não vêm instalados por padrão
    get_ipython().system('pip install -q ultralytics easyocr pytesseract')
    get_ipython().system('apt-get install -q -y tesseract-ocr')
    print("Tudo pronto!")

# TCC Experimento 1: Detecção de Veículos (UA-DETRAC)

**Objetivo**: Avaliar a resiliência e performance de YOLOv8, SSD e Faster R-CNN em diferentes cenários simulados.

---
**Métricas**: Precision, Recall, F1, mAP, FPS.

## 1. Configuração e Ambiente

In [ ]:
import sys, cv2, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display
import torch

# Configuração de caminhos dinâmicos
BASE_DIR = Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
%matplotlib inline

DS_PROC        = BASE_DIR / "dataset_processado"
UA_ROOT        = BASE_DIR / "UA-DETRAC"
RESULTS_DIR    = DS_PROC / "resultados"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

N_SAMPLES = 50   # Ajuste conforme necessário

device = "GPU" if torch.cuda.is_available() else "CPU"
print(f"Dispositivo : {device}")
print(f"UA-DETRAC   : {'Encontrado' if UA_ROOT.exists() else 'NÃO ENCONTRADO'}")

## 2. Análise do Dataset UA-DETRAC

In [ ]:
# =============================================================================
# CÓDIGO AUXILIAR (DETECTORES, MÉTRICAS E RUNNER) - CONSOLIDADO PARA COLAB
# =============================================================================
import torch
import cv2
import numpy as np
import pandas as pd
import time
import random
from pathlib import Path
from abc import ABC, abstractmethod
from ultralytics import YOLO
from torchvision.models.detection import fasterrcnn_resnet50_fpn, ssdlite320_mobilenet_v3_large
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights, SSDLite320_MobileNet_V3_Large_Weights
from tqdm import tqdm

# --- DETECTORES DE VEÍCULOS ---
class BaseDetector(ABC):
    @abstractmethod
    def detect(self, frame):
        """Retorna lista de detecções: [{'bbox': [x1, y1, x2, y2], 'conf': 0.9, 'class': 'car'}]"""
        pass

class YOLODetector(BaseDetector):
    def __init__(self, model_path='yolov8n.pt'):
        self.model = YOLO(model_path)
        self.model_name = "YOLOv8"
        
    def detect(self, frame):
        results = self.model(frame, verbose=False)[0]
        detections = []
        for box in results.boxes:
            detections.append({
                'bbox': box.xyxy[0].tolist(),
                'conf': float(box.conf),
                'class': self.model.names[int(box.cls)]
            })
        return detections

class TorchvisionDetector(BaseDetector):
    def __init__(self, model_type='faster_rcnn', confidence_threshold=0.5):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.threshold = confidence_threshold
        self.model_name = "Faster R-CNN" if model_type == 'faster_rcnn' else "SSD"
        
        if model_type == 'faster_rcnn':
            weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
            self.model = fasterrcnn_resnet50_fpn(weights=weights).to(self.device)
            self.classes = weights.meta["categories"]
        else: # SSD
            weights = SSDLite320_MobileNet_V3_Large_Weights.DEFAULT
            self.model = ssdlite320_mobilenet_v3_large(weights=weights).to(self.device)
            self.classes = weights.meta["categories"]
            
        self.model.eval()

    def detect(self, frame):
        # Preprocess
        img_tensor = torch.from_numpy(frame).permute(2, 0, 1).float().div(255).unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            prediction = self.model(img_tensor)[0]
        
        detections = []
        for i in range(len(prediction['boxes'])):
            score = float(prediction['scores'][i])
            if score > self.threshold:
                detections.append({
                    'bbox': prediction['boxes'][i].tolist(),
                    'conf': score,
                    'class': self.classes[int(prediction['labels'][i])]
                })
        return detections

class PlateDetector(BaseDetector):
    def __init__(self, model_path='yolov8n-plate.pt'):
        try:
            # Tenta carregar o modelo YOLO especializado
            self.model = YOLO(model_path)
            self.has_model = True
        except Exception as e:
            print(f"Aviso: Modelo YOLO de placas não encontrado. Usando fallback OpenCV.")
            self.has_model = False
        self.model_name = "PlateDetector"
        
    def detect(self, frame):
        if self.has_model:
            results = self.model(frame, verbose=False)[0]
            detections = []
            for box in results.boxes:
                detections.append({
                    'bbox': box.xyxy[0].tolist(),
                    'conf': float(box.conf),
                    'class': 'plate'
                })
            return detections
        else:
            # --- FALLBACK: Visão Computacional Clássica ---
            # Ideal para TCC: Detecção baseada em bordas e contornos
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            # Filtro para reduzir ruído mantendo bordas
            bfilter = cv2.bilateralFilter(gray, 11, 17, 17)
            # Detecção de bordas
            edged = cv2.Canny(bfilter, 30, 200)
            
            # Encontrar contornos
            keypoints = cv2.findContours(edged.copy(), cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
            contours = sorted(keypoints[0], key=cv2.contourArea, reverse=True)[:10]
            
            detections = []
            for res in contours:
                approx = cv2.approxPolyDP(res, 10, True)
                if len(approx) == 4: # Retângulos (possíveis placas)
                    x, y, w, h = cv2.boundingRect(res)
                    aspect_ratio = w / float(h)
                    # Placas brasileiras têm proporção de ~3:1 a ~4:1
                    if 2.0 < aspect_ratio < 5.0:
                        detections.append({
                            'bbox': [float(x), float(y), float(x+w), float(y+h)],
                            'conf': 0.8, # Confiança fixa para o fallback
                            'class': 'plate'
                        })
            return detections

# --- FUNÇÕES DE MÉTRICAS ---
def iou_single(box_a: list, box_b: list) -> float:
    """IoU entre dois boxes [x1, y1, x2, y2]."""
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    if inter == 0:
        return 0.0
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    return inter / (area_a + area_b - inter)


def match_detections(pred_boxes: list, gt_boxes: list,
                     iou_threshold: float = 0.5) -> tuple:
    """
    Faz o matching entre predições e ground-truth usando IoU.

    Retorna:
        tp (int): true positives
        fp (int): false positives
        fn (int): false negatives
    """
    if not gt_boxes:
        return 0, len(pred_boxes), 0
    if not pred_boxes:
        return 0, 0, len(gt_boxes)

    matched_gt = set()
    tp = 0
    fp = 0

    for pb in pred_boxes:
        best_iou = 0.0
        best_gt_idx = -1
        for gi, gb in enumerate(gt_boxes):
            if gi in matched_gt:
                continue
            iou = iou_single(pb, gb)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gi

        if best_iou >= iou_threshold and best_gt_idx >= 0:
            tp += 1
            matched_gt.add(best_gt_idx)
        else:
            fp += 1

    fn = len(gt_boxes) - len(matched_gt)
    return tp, fp, fn


# ─────────────────────────────────────────────────────────────────────────────
# Precision / Recall / F1
# ─────────────────────────────────────────────────────────────────────────────

def compute_precision(tp: int, fp: int) -> float:
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0


def compute_recall(tp: int, fn: int) -> float:
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0


def compute_f1(precision: float, recall: float) -> float:
    return (2 * precision * recall / (precision + recall)
            if (precision + recall) > 0 else 0.0)


# ─────────────────────────────────────────────────────────────────────────────
# AP (Average Precision) — interpolação PASCAL VOC 11 pontos
# ─────────────────────────────────────────────────────────────────────────────

def compute_ap(precisions: list, recalls: list) -> float:
    """
    Calcula AP usando interpolação de 11 pontos (PASCAL VOC).

    Args:
        precisions: lista de valores de precision em ordem crescente de recall
        recalls   : lista de valores de recall correspondentes

    Returns:
        AP (float 0–1)
    """
    ap = 0.0
    for threshold in np.arange(0.0, 1.1, 0.1):
        prec_at_rec = [p for p, r in zip(precisions, recalls) if r >= threshold]
        ap += max(prec_at_rec) if prec_at_rec else 0.0
    return ap / 11.0


def compute_ap_from_matches(tp_list: list, fp_list: list, n_gt: int) -> float:
    """
    Calcula AP a partir de listas cumulativas de TP e FP (ordenadas por confiança).

    Args:
        tp_list : lista de 0/1 indicando TP para cada detecção (ordem conf desc)
        fp_list : lista de 0/1 indicando FP para cada detecção
        n_gt    : total de ground-truth boxes

    Returns:
        AP (float 0–1)
    """
    if n_gt == 0:
        return 0.0

    tp_cum = np.cumsum(tp_list)
    fp_cum = np.cumsum(fp_list)

    recalls    = tp_cum / n_gt
    precisions = tp_cum / (tp_cum + fp_cum + 1e-9)

    # Adiciona pontos sentinela
    recalls    = np.concatenate(([0.0], recalls,    [recalls[-1]  if len(recalls)  else 0.0]))
    precisions = np.concatenate(([1.0], precisions, [0.0]))

    # Torna a curva monotonicamente decrescente
    for i in range(len(precisions) - 2, -1, -1):
        precisions[i] = max(precisions[i], precisions[i + 1])

    # Área sob a curva (mudanças de recall)
    idx = np.where(recalls[1:] != recalls[:-1])[0]
    ap  = np.sum((recalls[idx + 1] - recalls[idx]) * precisions[idx + 1])
    return float(ap)


# ─────────────────────────────────────────────────────────────────────────────
# Métricas agregadas para um conjunto de imagens
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_batch(results: list, iou_threshold: float = 0.5) -> dict:
    """
    Avalia um lote de imagens com predições e ground-truth.

    Args:
        results: lista de dicts com chaves:
            'pred_boxes' : [[x1,y1,x2,y2], ...]  (coordenadas absolutas)
            'gt_boxes'   : [[x1,y1,x2,y2], ...]
            'time_s'     : float (tempo de inferência em segundos)
            'conf_scores': [float, ...]            (confiança de cada pred)
        iou_threshold: limiar de IoU para considerar TP

    Returns:
        dict com precision, recall, f1, ap, fps, mean_conf
    """
    total_tp = total_fp = total_fn = 0
    all_tp_flags = []
    all_fp_flags = []
    n_gt_total   = 0
    total_time   = 0.0
    all_confs    = []

    for r in results:
        pred = r.get('pred_boxes', [])
        gt   = r.get('gt_boxes',   [])
        tp, fp, fn = match_detections(pred, gt, iou_threshold)
        total_tp += tp
        total_fp += fp
        total_fn += fn
        n_gt_total += len(gt)
        total_time += r.get('time_s', 0.0)
        all_confs.extend(r.get('conf_scores', []))

        # Para AP: 1 TP flag por predição
        for pb in pred:
            best = max((iou_single(pb, gb) for gb in gt), default=0.0)
            all_tp_flags.append(1 if best >= iou_threshold else 0)
            all_fp_flags.append(0 if best >= iou_threshold else 1)

    precision = compute_precision(total_tp, total_fp)
    recall    = compute_recall(total_tp, total_fn)
    f1        = compute_f1(precision, recall)
    ap        = compute_ap_from_matches(all_tp_flags, all_fp_flags, n_gt_total)
    fps       = len(results) / total_time if total_time > 0 else 0.0
    mean_conf = float(np.mean(all_confs)) if all_confs else 0.0

    return {
        'precision' : round(precision, 4),
        'recall'    : round(recall,    4),
        'f1'        : round(f1,        4),
        'ap'        : round(ap,        4),
        'fps'       : round(fps,       2),
        'mean_conf' : round(mean_conf, 4),
        'tp'        : total_tp,
        'fp'        : total_fp,
        'fn'        : total_fn,
        'n_frames'  : len(results),
    }


# ─────────────────────────────────────────────────────────────────────────────
# ExperimentLogger — registro frame a frame
# ─────────────────────────────────────────────────────────────────────────────


# --- UA-DETRAC RUNNER ---
# Parser de labels YOLO → bboxes absolutas
# ─────────────────────────────────────────────────────────────────────────────

def _yolo_to_abs(cx: float, cy: float, w: float, h: float,
                 img_w: int, img_h: int) -> list:
    """Converte bbox YOLO normalizada para [x1, y1, x2, y2] absoluto."""
    x1 = (cx - w / 2) * img_w
    y1 = (cy - h / 2) * img_h
    x2 = (cx + w / 2) * img_w
    y2 = (cy + h / 2) * img_h
    return [x1, y1, x2, y2]


def _load_gt_boxes(label_path: Path, img_w: int, img_h: int) -> list:
    """
    Lê um arquivo de label YOLO e retorna a lista de bboxes absolutas.
    Classe 1 = veículo (UA-DETRAC usa classe 0 e 1 para veículos).
    """
    if not label_path.exists():
        return []
    boxes = []
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            # cls = int(parts[0])  # todas as classes são veículos no UA-DETRAC
            cx, cy, bw, bh = map(float, parts[1:5])
            boxes.append(_yolo_to_abs(cx, cy, bw, bh, img_w, img_h))
    return boxes


# ─────────────────────────────────────────────────────────────────────────────
# Indexador: mapeia nome de frame → caminho do label no UA-DETRAC
# ─────────────────────────────────────────────────────────────────────────────

def _build_label_index(ua_detrac_root: Path) -> dict:
    """
    Cria um índice {stem_do_frame: Path_do_label} percorrendo
    UA-DETRAC/DETRAC_Upload/labels/train e val.
    """
    index = {}
    labels_root = ua_detrac_root / 'DETRAC_Upload' / 'labels'
    for split_dir in labels_root.iterdir():
        if not split_dir.is_dir():
            continue
        # Tenta layout flat (arquivos de label diretamente sob o split)
        for lbl in split_dir.glob('*.txt'):
            index[lbl.stem] = lbl
        # Tenta layout nested (arquivos de label agrupados em subpastas de sequências)
        for seq_dir in split_dir.iterdir():
            if seq_dir.is_dir():
                for lbl in seq_dir.glob('*.txt'):
                    index[lbl.stem] = lbl
    return index


# ─────────────────────────────────────────────────────────────────────────────
# Função principal de benchmark
# ─────────────────────────────────────────────────────────────────────────────

VEHICLE_CLASSES = {'car', 'truck', 'bus', 'motorcycle', 'van'}


def run_ua_detrac_benchmark(
    scenario_dirs: dict,
    models_config: list,
    ua_detrac_root: Path,
    n_samples: int = 100,
    iou_threshold: float = 0.5,
    seed: int = 42,
) -> pd.DataFrame:
    """
    Executa o benchmark de detecção de veículos com o UA-DETRAC.

    Args:
        scenario_dirs : dict {'diurno': Path, 'noturno': Path, 'baixa_qualidade': Path}
        models_config : lista de (nome, detector_instance)
        ua_detrac_root: pasta raiz do UA-DETRAC (contém DETRAC_Upload/)
        n_samples     : número de imagens por cenário (para viabilizar execução)
        iou_threshold : limiar de IoU para TP
        seed          : semente para reprodutibilidade da amostragem

    Returns:
        DataFrame com uma linha por (modelo, cenário, frame)
    """
    random.seed(seed)

    print("Indexando labels do UA-DETRAC...")
    label_index = _build_label_index(ua_detrac_root)
    print(f"  {len(label_index)} labels indexados.")

    all_rows = []

    for model_name, detector in models_config:
        for scenario_name, scenario_dir in scenario_dirs.items():
            # Coleta imagens do cenário
            img_paths = list(scenario_dir.rglob('*.jpg')) + \
                        list(scenario_dir.rglob('*.png'))

            if not img_paths:
                print(f"  [AVISO] Nenhuma imagem em {scenario_dir}")
                continue

            # Amostragem aleatória
            sample = random.sample(img_paths, min(n_samples, len(img_paths)))

            results_batch = []

            for img_path in tqdm(sample,
                                 desc=f'{model_name} | {scenario_name}',
                                 leave=False):
                img = cv2.imread(str(img_path))
                if img is None:
                    continue
                h_img, w_img = img.shape[:2]

                # Ground truth via label index
                gt_boxes = label_index.get(img_path.stem, None)
                if gt_boxes is not None:
                    gt_boxes = _load_gt_boxes(gt_boxes, w_img, h_img)
                else:
                    gt_boxes = []

                # Inferência
                t0 = time.perf_counter()
                detections = detector.detect(img)
                elapsed = time.perf_counter() - t0

                # Filtra só veículos
                pred_boxes   = []
                conf_scores  = []
                for d in detections:
                    if d['class'].lower() in VEHICLE_CLASSES:
                        pred_boxes.append(d['bbox'])
                        conf_scores.append(d['conf'])

                results_batch.append({
                    'pred_boxes' : pred_boxes,
                    'gt_boxes'   : gt_boxes,
                    'time_s'     : elapsed,
                    'conf_scores': conf_scores,
                })

                # Linha por frame
                tp, fp, fn = match_detections(pred_boxes, gt_boxes, iou_threshold)
                fps  = 1.0 / elapsed if elapsed > 0 else 0.0
                prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
                rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
                f1   = (2 * prec * rec / (prec + rec)
                        if (prec + rec) > 0 else 0.0)

                all_rows.append({
                    'model'    : model_name,
                    'scenario' : scenario_name,
                    'image'    : img_path.name,
                    'n_pred'   : len(pred_boxes),
                    'n_gt'     : len(gt_boxes),
                    'tp'       : tp,
                    'fp'       : fp,
                    'fn'       : fn,
                    'precision': round(prec, 4),
                    'recall'   : round(rec,  4),
                    'f1'       : round(f1,   4),
                    'fps'      : round(fps,  2),
                    'confidence': round(float(np.mean(conf_scores))
                                        if conf_scores else 0.0, 4),
                    'time_ms'  : round(elapsed * 1000, 2),
                })

    return pd.DataFrame(all_rows)


def compute_map_from_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula mAP por (modelo, cenário) a partir do DataFrame de resultados frame a frame.

    Usa as colunas: tp, fp, n_gt por imagem.
    AP é calculado via AUC da curva Precision-Recall.
    """

    records = []
    for (model, scenario), grp in df.groupby(['model', 'scenario']):
        # Para AP: ordena por confiança desc e acumula TP/FP
        grp_sorted = grp.sort_values('confidence', ascending=False)
        tp_flags = []
        fp_flags = []
        for _, row in grp_sorted.iterrows():
            # Simplificação: cada frame com tp>0 contribui com tp TPs e fp FPs
            tp_flags.extend([1] * row['tp'] + [0] * row['fp'])
            fp_flags.extend([0] * row['tp'] + [1] * row['fp'])

        n_gt = grp['n_gt'].sum()
        ap   = compute_ap_from_matches(tp_flags, fp_flags, n_gt)

        records.append({
            'model'   : model,
            'scenario': scenario,
            'AP'      : round(ap, 4),
        })

    map_df = pd.DataFrame(records)
    overall = map_df.groupby('model')['AP'].mean().reset_index()
    overall['scenario'] = 'GERAL (mAP)'
    overall.rename(columns={'AP': 'AP'}, inplace=True)

    return pd.concat([map_df, overall], ignore_index=True)


In [ ]:
SCENARIOS = {
    "Diurno"         : DS_PROC / "diurno",
    "Noturno"        : DS_PROC / "noturno",
    "Baixa Qualidade": DS_PROC / "baixa_qualidade",
}

counts = {}
for name, path in SCENARIOS.items():
    imgs = list(path.glob("*.jpg")) + list(path.glob("*.png"))
    counts[name] = len(imgs)

total = sum(counts.values())
print(f"Total de imagens: {total:,}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(counts.keys(), counts.values(), color=["#4C72B0", "#404040", "#DD8452"])
ax.set_title(f"Distribuição UA-DETRAC (Total: {total:,})")
plt.show()

## 3. Experimentos de Detecção

In [ ]:

models = {
    "YOLOv8": YOLODetector("yolov8n.pt"),
    "SSD": TorchvisionDetector("ssd"),
    "Faster R-CNN": TorchvisionDetector("faster_rcnn")
}


df_ua = run_ua_detrac_benchmark(
    scenario_dirs  = SCENARIOS,
    models_config  = list(models.items()),
    ua_detrac_root = UA_ROOT,
    n_samples      = N_SAMPLES,
)
df_ua.to_csv(RESULTS_DIR / "ua_detrac_benchmark.csv", index=False)
display(df_ua.head())

## 4. Resumo de Resultados

In [ ]:
summary_ua = df_ua.groupby(["model", "scenario"]).agg(
    Precision = ("precision", "mean"),
    Recall    = ("recall",    "mean"),
    F1        = ("f1",        "mean"),
    FPS       = ("fps",       "mean")
).round(3)

display(summary_ua)

map_df = compute_map_from_df(df_ua)
display(map_df[map_df["scenario"] == "GERAL (mAP)"])